# ST-Adaptive-Ensemble v3 – Huấn luyện từ đầu trên Kaggle (2 × GPU T4)

Mọi mô hình được huấn luyện lại bằng cùng mã và cùng môi trường thư viện. Kết quả được **push lên nhánh `run` sau từng run hoàn tất**.

| Đợt | `RUN_IDS` | `ENSEMBLE_RUN_IDS` | `FRESH_START` | `FREEZE_CHAMPION` |
|---|---|---|---|---|
| **1** | `"0-4"` (seed 42–46) | `"0-4"` | `True` | `True` |
| **2** | `"5-9"` (seed 47–51) | `"0-9"` | `False` | `False` |

- **Song song:** GPU 0 chạy ST-WaveFormer ABILENE; GPU 1 chạy ST-WaveFormer SDN rồi GÉANT (epoch 200, patience 30); CPU chạy XGBoost, LightGBM, CatBoost, LightGBM-Residual.
- **Push theo run:** một luồng nền cứ `PUSH_EVERY_MIN` phút lại commit và push các run **đã hoàn tất** lên nhánh `run`. Run đang chạy dở không bị đưa lên.
- **Tiếp tục khi bị ngắt:** nếu nhánh `run` đã có trên GitHub, notebook clone từ nhánh đó và **không xóa** log. Nhờ `--skip_existing`, chỉ các run còn thiếu được huấn luyện. Đợt 2 cũng tự nối tiếp trên nhánh `run`.
- **Muốn làm lại đợt 1 từ đầu:** xóa nhánh `run` trên GitHub trước khi chạy.

**Trước khi chạy:** *Settings → Accelerator →* **GPU T4 x2**; *Internet →* **On**; gắn secret `GITHUB_TOKEN` (có quyền ghi repo); chạy nền bằng **Save & Run All (Commit)**.

## 0. Cấu hình

In [ ]:
import os

REPO_SLUG     = "Vannt20/network_prediction_v2"   # owner/repo trên GitHub
BASE_BRANCH   = "main"     # nhánh chứa mã; chỉ dùng khi nhánh PUSH_BRANCH chưa tồn tại
PUSH_BRANCH   = "run"      # nhánh nhận kết quả (push sau từng run)
SECRET_NAME   = "GITHUB_TOKEN"
GIT_USER_NAME = "Van207"
GIT_USER_MAIL = "Nguyenthevan207@gmail.com"

WORKDIR = "/kaggle/working/network_prediction_v2"

# ---- ĐỢT 1 ----
RUN_IDS          = "0-4"    # run huấn luyện trong đợt này (seed = 42 + run_id)
ENSEMBLE_RUN_IDS = "0-4"    # run đưa vào tổng hợp ensemble (đợt 2: "0-9")
FRESH_START      = True     # xóa logs/, cache/, results/ khi bắt đầu mới (bị bỏ qua nếu tiếp tục từ nhánh run)
FREEZE_CHAMPION  = True     # bầu champion trên RUN_IDS rồi cố định (đợt 2: False)
# ----------------

EPOCHS    = 200
PATIENCE  = 30
ML_MODELS = "xgboost,lightgbm,catboost,lightgbm_res"   # 3 ứng viên champion + nhánh 3

GPU_PLAN = {0: ["abilene"], 1: ["sdn", "geant"]}
PUSH_EVERY_MIN = 10        # chu kỳ quét và push các run đã hoàn tất
print("Push branch:", PUSH_BRANCH, "| RUN_IDS:", RUN_IDS, "| FRESH_START:", FRESH_START)

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("CUDA:", torch.cuda.is_available(), "| Số GPU:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, "Cần bật Accelerator = GPU T4 x2"

## 2. Xác thực GitHub bằng token (không ghi token ra đĩa)

Token được truyền qua header HTTP ở từng lệnh `git` (`-c http.extraheader`). Vì vậy token **không** nằm trong `.git/config` hay remote URL, và không bị lưu lại trong output của notebook khi Save Version.

In [ ]:
import base64, subprocess, shlex
from kaggle_secrets import UserSecretsClient

_TOKEN = UserSecretsClient().get_secret(SECRET_NAME)
assert _TOKEN, f"Không đọc được secret {SECRET_NAME}"
_AUTH = base64.b64encode(f"x-access-token:{_TOKEN}".encode()).decode()
_AUTH_ARGS = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {_AUTH}"]


def git(*args, cwd=None, auth=False, check=True, quiet=False):
    # Chạy git; che token trong mọi output/lỗi.
    cmd = ["git"] + (_AUTH_ARGS if auth else []) + list(args)
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_TOKEN, "***").replace(_AUTH, "***")
    if not quiet or r.returncode != 0:
        print(f"$ git {' '.join(args)}\n{out.strip()}")
    if check and r.returncode != 0:
        raise RuntimeError(f"git {args[0]} thất bại (exit {r.returncode})")
    return r


git("config", "--global", "user.name", GIT_USER_NAME)
git("config", "--global", "user.email", GIT_USER_MAIL)
git("config", "--global", "http.postBuffer", "524288000")
REPO_URL = f"https://github.com/{REPO_SLUG}.git"
git("ls-remote", "--heads", REPO_URL, BASE_BRANCH, auth=True)

## 3. Clone repository (tự tiếp tục từ nhánh `run` nếu đã có)

In [ ]:
RESUMED = bool(git("ls-remote", "--heads", REPO_URL, PUSH_BRANCH, auth=True, quiet=True).stdout.strip())
if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    src_branch = PUSH_BRANCH if RESUMED else BASE_BRANCH
    git("clone", "--depth", "1", "--branch", src_branch, REPO_URL, WORKDIR, auth=True)
else:
    print("Repo đã tồn tại, bỏ qua clone.")

os.chdir(WORKDIR)
git("checkout", "-B", PUSH_BRANCH, cwd=WORKDIR)
git("log", "--oneline", "-3", cwd=WORKDIR)
print("TIẾP TỤC từ nhánh", PUSH_BRANCH if RESUMED else f"-> BẮT ĐẦU MỚI từ {BASE_BRANCH}")

## 4. Cài thư viện

Không cài lại `requirements.txt` vì file này có tensorflow và torch, cài lại dễ làm hỏng bản CUDA sẵn có của Kaggle. Chỉ cài bổ sung những gói còn thiếu.

In [ ]:
import importlib, subprocess, sys
missing = [p for p in ["lightgbm", "catboost", "xgboost", "openpyxl", "seaborn"]
           if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import lightgbm, catboost, xgboost
print("lightgbm", lightgbm.__version__, "| catboost", catboost.__version__, "| xgboost", xgboost.__version__)

## 5. Dọn log cũ (chỉ khi bắt đầu mới và `FRESH_START = True`)

Khi tiếp tục từ nhánh `run`, log đã push được giữ nguyên. Thư mục `data/` không bị động tới.

In [ ]:
import glob, shutil, json, time
import pandas as pd

SEQ = {"sdn": 60, "geant": 24, "abilene": 24}
if FRESH_START and not RESUMED:
    for d in ["logs", "results", "cache"]:
        if os.path.isdir(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)
    print("Bắt đầu mới: đã xóa logs/, results/, cache/ trong bản clone")
else:
    print("Giữ log hiện có:", sorted(os.listdir("logs")) if os.path.isdir("logs") else "(trống)")
os.makedirs("logs", exist_ok=True); os.makedirs("results", exist_ok=True); os.makedirs("cache/v3", exist_ok=True)

if not FREEZE_CHAMPION:   # đợt 2: bắt buộc có champion đã cố định từ đợt 1
    champ = "results/champion_ml_model.json"
    assert os.path.exists(champ), "Đợt 2 cần results/champion_ml_model.json từ đợt 1 (nhánh run)"
    info = json.load(open(champ, encoding="utf-8"))
    print("Champion:", info["champion_model"], "| frozen:", info.get("frozen"), "| bầu trên run:", info.get("selected_on_runs"))
    assert info.get("frozen"), "Champion của đợt 1 chưa được cố định"

## 6. Push theo từng run lên nhánh `run`

Một run được coi là **hoàn tất** khi file cuối cùng của nó đã được ghi và không đổi ít nhất `MIN_AGE_S` giây:

| Loại log | File đánh dấu |
|---|---|
| `logs/stwaveformer_*/run_*` | `y_pred_data_raw.npy` |
| `logs/*_shared/run_*` (học máy) | `model.bin` |
| `logs/st_adaptive_ensemble_*/run_*` | `test_metrics.csv` |
| `cache/v3/*.pt`, `results/*` | chính file đó |

File > 95 MB bị bỏ qua (giới hạn GitHub là 100 MB). Mọi thao tác git dùng chung một khóa, nên luồng nền và các ô push thủ công không chạy chồng lên nhau.

In [ ]:
import threading

GIT_LOCK = threading.Lock()
MIN_AGE_S = 60
MAX_MB = 95


def _stable(f):
    return os.path.exists(f) and time.time() - os.path.getmtime(f) > MIN_AGE_S


def completed_paths():
    paths = []
    for d in glob.glob("logs/*/run_*"):
        group = os.path.basename(os.path.dirname(d))
        if group.startswith("stwaveformer_"):
            marker = "y_pred_data_raw.npy"
        elif group.endswith("_shared"):
            marker = "model.bin"
        else:
            marker = "test_metrics.csv"
        if _stable(os.path.join(d, marker)):
            paths.append(d)
    paths += [f for f in glob.glob("cache/v3/*.pt") + glob.glob("results/*") if os.path.isfile(f) and _stable(f)]
    return paths


def _too_big(path):
    files = [path] if os.path.isfile(path) else [os.path.join(r, f) for r, _, fs in os.walk(path) for f in fs]
    return [f for f in files if os.path.getsize(f) > MAX_MB * 1024 ** 2]


def push_completed(label="tự động"):
    with GIT_LOCK:
        paths = [p for p in completed_paths() if not _too_big(p)]
        for p in paths:
            git("add", "-A", p, cwd=WORKDIR, quiet=True)
        staged = git("diff", "--cached", "--name-only", cwd=WORKDIR, quiet=True).stdout.split()
        if not staged:
            return 0
        # logs/<nhóm>/run_<k>/<file> -> logs/<nhóm>/run_<k>; file khác giữ nguyên
        runs = sorted({"/".join(f.split("/")[:3]) if f.startswith("logs/") else f for f in staged})
        msg = f"Kaggle v3 ({label}): {len(runs)} mục hoàn tất\n\n" + "\n".join(runs[:60])
        git("commit", "-q", "-m", msg, cwd=WORKDIR, quiet=True)
        for attempt in range(3):
            r = git("push", "-u", "origin", PUSH_BRANCH, cwd=WORKDIR, auth=True, check=False, quiet=True)
            if r.returncode == 0:
                print(f"[push {time.strftime('%H:%M')}] {len(runs)} mục -> {PUSH_BRANCH}: {', '.join(runs[:8])}{' ...' if len(runs) > 8 else ''}")
                return len(runs)
            time.sleep(20 * (attempt + 1))
        print("[push] thất bại sau 3 lần thử, sẽ thử lại ở chu kỳ sau")
        return 0


class RunPusher(threading.Thread):
    def __init__(self, every_min=PUSH_EVERY_MIN):
        super().__init__(daemon=True)
        self.every = every_min * 60
        self._stop_evt = threading.Event()

    def run(self):
        while not self._stop_evt.wait(self.every):
            try:
                push_completed()
            except Exception as e:
                print(f"[push] lỗi: {e}")

    def stop(self):
        self._stop_evt.set()
        self.join(timeout=10)
        global MIN_AGE_S
        old, MIN_AGE_S = MIN_AGE_S, 0      # tiến trình đã kết thúc -> mọi file đã ghi xong
        try:
            push_completed("cuối giai đoạn")
        finally:
            MIN_AGE_S = old


def commit_and_push(message):
    # Push toàn bộ logs/results/cache (dùng sau khi mọi tiến trình đã kết thúc).
    with GIT_LOCK:
        big = [f for d in ("logs", "results", "cache") if os.path.exists(d) for f in _too_big(d)]
        assert not big, f"File > {MAX_MB}MB, GitHub sẽ từ chối: {big}"
        git("add", "-A", "logs", "results", "cache", cwd=WORKDIR)
        if git("diff", "--cached", "--quiet", cwd=WORKDIR, check=False).returncode != 0:
            git("commit", "-m", message, cwd=WORKDIR)
        git("push", "-u", "origin", PUSH_BRANCH, cwd=WORKDIR, auth=True)

## 6b. Bước A: ST-WaveFormer trên 2 GPU + 4 mô hình học máy trên CPU (song song)

`--skip_existing` chỉ bỏ qua run hợp lệ đã có. Log được ghi vào `/kaggle/working/logs_kaggle/`.

In [ ]:
import subprocess, time, sys

LOG_DIR = "/kaggle/working/logs_kaggle"
os.makedirs(LOG_DIR, exist_ok=True)
PY = sys.executable


def launch(name, shell_cmd, gpu=None):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "" if gpu is None else str(gpu)
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"
    log = open(os.path.join(LOG_DIR, f"{name}.log"), "w")
    p = subprocess.Popen(shell_cmd, shell=True, cwd=WORKDIR, env=env,
                         stdout=log, stderr=subprocess.STDOUT)
    print(f"[launch] {name} (GPU={gpu}) pid={p.pid}\n   {shell_cmd}")
    return name, p


def wait_all(procs, every=120, tail=3):
    t0 = time.time()
    while True:
        alive = [(n, p) for n, p in procs if p.poll() is None]
        mins = (time.time() - t0) / 60
        print(f"\n===== {mins:.1f} phút | còn chạy: {[n for n, _ in alive]} =====")
        for n, _ in procs:
            try:
                lines = open(os.path.join(LOG_DIR, f"{n}.log"), encoding="utf-8", errors="ignore").read().splitlines()
                for l in lines[-tail:]:
                    print(f"  [{n}] {l[:160]}")
            except FileNotFoundError:
                pass
        if not alive:
            break
        time.sleep(every)
    codes = {n: p.returncode for n, p in procs}
    print("\nExit codes:", codes)
    bad = [n for n, c in codes.items() if c != 0]
    assert not bad, f"Tiến trình lỗi: {bad} – xem {LOG_DIR}/<tên>.log"


stage_a = []
for gpu, dss in GPU_PLAN.items():
    cmd = " && ".join(
        f"{PY} run_experiments.py --model STWaveFormer --dataset {ds} "
        f"--run_ids {RUN_IDS} --epochs {EPOCHS} --patience {PATIENCE} --skip_existing"
        for ds in dss)
    stage_a.append(launch(f"stwaveformer_gpu{gpu}", cmd, gpu=gpu))

freeze = " --freeze_champion" if FREEZE_CHAMPION else ""
stage_a.append(launch(
    "ml_cpu",
    f"{PY} baselines_ml/run_ml_baselines.py --models {ML_MODELS} "
    f"--datasets sdn,geant,abilene --run_ids {RUN_IDS} --skip_existing{freeze}",
    gpu=None))

pusher = RunPusher()
pusher.start()
try:
    wait_all(stage_a)
finally:
    pusher.stop()   # push nốt các run vừa xong, kể cả khi có tiến trình lỗi

### Kiểm tra kết quả bước A

In [ ]:
# Kiểm tra trực tiếp trên logs/ (nguồn dữ liệu của cache và báo cáo)
from run_experiments import check_existing_run, expected_test_windows, parse_run_ids
ids = parse_run_ids(ENSEMBLE_RUN_IDS)
info = json.load(open("results/champion_ml_model.json", encoding="utf-8"))
print("Champion:", info["champion_model"], "| frozen:", info.get("frozen"), "| bầu trên run:", info.get("selected_on_runs"))
ok = bool(info.get("frozen"))
for ds, sl in SEQ.items():
    n_win = expected_test_windows(ds, sl)
    bad_dl = [r for r in ids if not check_existing_run(f"logs/stwaveformer_data_{ds}_seq_{sl}/run_{r}", n_win)[0]]
    miss_ml = [(m, r) for m in ML_MODELS.split(",") for r in ids
               if not os.path.exists(f"logs/{m}_data_{ds}_shared/run_{r}/model.bin")]
    print(f"{ds.upper():8s} ST-WaveFormer lỗi/thiếu: {bad_dl or 'không'} | ML thiếu: {miss_ml or 'không'}")
    ok &= not bad_dl and not miss_ml
assert ok, "Còn run thiếu hoặc champion chưa cố định – kiểm tra log trước khi đi tiếp"

## 7. Push checkpoint sau Bước A

Các run đã được push dần trong lúc huấn luyện; bước này đẩy nốt phần còn lại (ví dụ `results/champion_ml_model.json`).

In [ ]:
commit_and_push(f"Kaggle v3: hoàn tất huấn luyện run {RUN_IDS} (ST-WaveFormer epoch 200/patience 30 + ML)")

## 8. Bước B: Cache v3, Stacking, Ablation, Báo cáo

`--skip_dl`: không huấn luyện thêm. Chạy trên cùng môi trường với bước huấn luyện, nên phiên bản thư viện khi nạp lại mô hình là thống nhất.

In [ ]:
pusher = RunPusher()
pusher.start()
try:
    wait_all([launch("ensemble_v3",
                     f"{PY} training/run_ensemble.py --datasets all --run_ids {ENSEMBLE_RUN_IDS} --skip_dl",
                     gpu=0)], every=60, tail=5)
finally:
    pusher.stop()

In [ ]:
print(open("results/bang_tong_hop_luan_van.csv", encoding="utf-8-sig").read())

### Tóm tắt kết quả mới

In [ ]:
pd.set_option("display.width", 200)
print(pd.read_csv("results/bang_tong_hop_luan_van.csv").to_string(index=False))
print()
print(pd.read_csv("results/kiem_dinh_thong_ke.csv").round(4).to_string(index=False))
print()
print(pd.read_csv("results/ablation_summary.csv").drop(columns=["description"]).round(4).to_string(index=False))

## 9. Commit và push kết quả cuối

In [ ]:
commit_and_push(f"Kaggle v3: ensemble cho run {ENSEMBLE_RUN_IDS}")
print(f"\nĐã push lên: https://github.com/{REPO_SLUG}/tree/{PUSH_BRANCH}")

## 10. Kéo kết quả về máy local

Kết quả nằm trên nhánh `run` (mã của `main` + các commit kết quả). Trên máy local (thư mục đã nối với `origin`):

```bash
git fetch origin run
git switch -c run --track origin/run
python preflight_check.py --run_ids 0-4
```

Nhánh `main` vẫn chỉ chứa mã. Khi cập nhật mã trên `main` giữa hai đợt, cần merge `main` vào `run` trước khi chạy đợt 2 (`git switch run && git merge main && git push`).